# Myanmar Conflict Analysis

Consolidates four scripts into a single reproducible pipeline.

| Step | Source script | What it does | Key outputs |
|------|--------------|--------------|-------------|
| 3 | `03_myanmar_actor_analysis.py` | Actor ranking by importance score across 4 political periods | PNGs, CSVs, ranking report |
| 4 | `04_myanmar_gif.py` | 3-month rolling conflict map GIF by top-10 actor | `02_monthly_conflict.gif` |
| 5 | `05_kikuta_zoning.py` | OCSVM conflict-zoning per dyad × month (Kikuta 2022) | Parquet zone files |
| 6 | `06_kikuta_gif.py` | Animated zone GIF for 4 primary military dyads | `03_zones_*.gif`, `03_zones_combined.gif` |

**Run in order.** Step 3 must complete before Step 4 (actor colors). Step 4 downloads Natural Earth boundaries (~10 MB) on first run; Steps 5 and 6 read them from disk.

In [ ]:
import io
import json
import warnings
import zipfile
from datetime import date
from pathlib import Path
from typing import Optional

import geopandas as gpd
import imageio.v2 as iio
import matplotlib
matplotlib.use("Agg")
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import requests
import shapely.affinity as sa
from shapely import wkt
from shapely.geometry import box
from shapely.ops import unary_union
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(20260428)

In [ ]:
ROOT = Path("..").resolve()
DATA_IN = ROOT / "data" / "processed" / "acled_clean.parquet"
BOUNDARY_DIR = ROOT / "data" / "raw" / "boundaries" / "myanmar"
ZONES_DIR = ROOT / "data" / "processed" / "kikuta_zones"

GEO_THRESH = 2

MILITARY_VARIANTS = frozenset({
    "Military Forces of Myanmar (2021-)",
    "Military Forces of Myanmar (2016-2021)",
    "Military Forces of Myanmar (2011-2016)",
    "Military Forces of Myanmar (1988-2011)",
})
MILITARY_LABEL = "Military Forces of Myanmar (merged)"
PDF_LABEL = "People's Defense Force (merged)"

EXCLUDE = frozenset({
    "Unidentified Armed Group (Myanmar)",
    "Unidentified Anti-Coup Armed Group",
    "Protesters (Myanmar)",
    "Rioters (Myanmar)",
    "Police Forces of Myanmar (2021-)",
})

## Step 3 — Actor Ranking and Period Analysis

Myanmar's armed conflict spans four analytically distinct political periods defined by major structural breaks:

| Period | Dates | Key dynamic |
|--------|-------|-------------|
| P1 | 2010-01 / 2016-09 | Ceasefire-era; military engages EAOs in Kachin and Shan states |
| P2 | 2016-10 / 2021-01 | Rakhine crisis (Arakan Army, Rohingya displacement) and pre-coup escalation |
| P3 | 2021-02 / 2023-10 | February 2021 coup; civil war between military and resistance (PDF + EAOs) |
| P4 | 2023-11 / 2026-04 | Operation 1027 and coordinated rebel advances (Three Brotherhood Alliance) |

**Importance score** ranks actors jointly on events and fatalities to avoid two failure modes: a high-event / low-fatality actor (skirmishes) crowding out a high-fatality actor, and vice versa.

```
importance_score = (rank_by_events + rank_by_fatalities) / 2
```

Lower score → more important. Ties broken by fatalities descending.

### Actor Harmonization

ACLED records the same real-world organization under multiple labels when its political status changes. We consolidate these before ranking so trends are attributable to a single entity.

**Military Forces of Myanmar** appears in four date-suffix variants reflecting regime transitions (1988-2011, 2011-2016, 2016-2021, 2021-). All four are merged into one label.

**People's Defense Force (PDF)** is recorded at district level — `PDF: Sagaing Region`, `PDF: Mandalay Region`, etc. — because it formed as a decentralized resistance after the coup. All variants sharing the "People's Defense Force" prefix or starting with `PDF:` are merged.

**Excluded actors** (Unidentified Armed Group, Protesters, Rioters, Police Forces) are either non-attributable or represent categories outside the scope of armed-group conflict analysis.

In [ ]:
COUNTRY = "Myanmar"
IMPORTANCE_N = 10

PERIODS = [
    ("2010-01", "2016-09", "P1: Ceasefire-era ethnic conflict"),
    ("2016-10", "2021-01", "P2: Rakhine crisis & pre-coup escalation"),
    ("2021-02", "2023-10", "P3: Coup and civil war"),
    ("2023-11", "2026-04", "P4: Operation 1027 & rebel advances"),
]

PERIOD_DATE_RANGES = [
    (pd.Timestamp("2010-01-01"), pd.Timestamp("2016-09-30")),
    (pd.Timestamp("2016-10-01"), pd.Timestamp("2021-01-31")),
    (pd.Timestamp("2021-02-01"), pd.Timestamp("2023-10-31")),
    (pd.Timestamp("2023-11-01"), pd.Timestamp("2026-04-30")),
]

PERIOD_COLORS = ["#BBDEFB", "#FFE082", "#FFCDD2", "#C8E6C9"]
TAB10 = list(plt.cm.tab10.colors)

OUT_DIR_03 = ROOT / "output" / "myanmar"
OUT_DIR_03.mkdir(parents=True, exist_ok=True)

In [ ]:
def load_myanmar(path: Path) -> pd.DataFrame:
    df = pd.read_parquet(path)
    df = df[
        (df["country"] == COUNTRY) &
        (df["fatalities"] > 0) &
        (df["geo_precision"] <= GEO_THRESH)
    ].copy()
    df["event_date"] = pd.to_datetime(df["event_date"])
    return df


def apply_merges(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.loc[df["actor1"].isin(MILITARY_VARIANTS), "actor1"] = MILITARY_LABEL
    pdf_mask = (
        df["actor1"].str.contains("People's Defense Force", na=False)
        | df["actor1"].str.startswith("PDF:", na=False)
    )
    df.loc[pdf_mask, "actor1"] = PDF_LABEL
    return df


def apply_exclusions(df: pd.DataFrame) -> pd.DataFrame:
    return df[~df["actor1"].isin(EXCLUDE)].copy()


def importance_ranking(df: pd.DataFrame, top_n: int = IMPORTANCE_N):
    agg = (
        df.groupby("actor1")
        .agg(events=("event_id_cnty", "count"), fatalities=("fatalities", "sum"))
        .reset_index()
    )
    agg["rank_events"] = agg["events"].rank(ascending=False, method="min")
    agg["rank_fatalities"] = agg["fatalities"].rank(ascending=False, method="min")
    agg["importance_score"] = (agg["rank_events"] + agg["rank_fatalities"]) / 2
    agg = agg.sort_values(
        ["importance_score", "fatalities"], ascending=[True, False]
    ).reset_index(drop=True)
    agg["global_rank"] = agg.index + 1
    return agg, agg.head(top_n).copy()


def period_ranking(df: pd.DataFrame, top_n: int = 3) -> pd.DataFrame:
    df = df.copy()
    df["ym"] = df["event_date"].dt.to_period("M")
    records = []
    for start, end, label in PERIODS:
        sub = df[(df["ym"] >= start) & (df["ym"] <= end)]
        if sub.empty:
            continue
        agg = (
            sub.groupby("actor1")
            .agg(events=("event_id_cnty", "count"), fatalities=("fatalities", "sum"))
            .reset_index()
        )
        agg["rank_events"] = agg["events"].rank(ascending=False, method="min")
        agg["rank_fatalities"] = agg["fatalities"].rank(ascending=False, method="min")
        agg["importance_score"] = (agg["rank_events"] + agg["rank_fatalities"]) / 2
        agg = agg.sort_values(
            ["importance_score", "fatalities"], ascending=[True, False]
        ).reset_index(drop=True)
        for i, row in agg.head(top_n).iterrows():
            records.append({
                "period": label,
                "period_start": start,
                "period_end": end,
                "period_rank": i + 1,
                "actor1": row["actor1"],
                "events": int(row["events"]),
                "fatalities": int(row["fatalities"]),
                "importance_score": round(row["importance_score"], 1),
            })
    return pd.DataFrame(records)

### Visualization and Reporting

Two figures summarize the results:

- **Period proposal chart** — monthly events (bars) and fatalities (line) with period boundaries shaded; lets us validate that the chosen breakpoints align with observable conflict dynamics.
- **Actor timeline** — monthly fatal events per top-10 actor over the full period; reveals which actors drive each phase.

The ranking report (`01_actor_ranking.md`) combines the global top-10 table and per-period top-3 tables into a single markdown document for qualitative review.

In [ ]:
def plot_period_proposal(df_raw: pd.DataFrame, out_path: Path) -> None:
    monthly = (
        df_raw.groupby(df_raw["event_date"].dt.to_period("M"))
        .agg(events=("event_id_cnty", "count"), fatalities=("fatalities", "sum"))
        .reset_index()
    )
    monthly["date"] = monthly["event_date"].dt.to_timestamp()

    fig, ax1 = plt.subplots(figsize=(14, 5))
    ax2 = ax1.twinx()

    ax1.bar(monthly["date"], monthly["events"], width=25, color="#5B9BD5", alpha=0.7, zorder=2)
    ax2.plot(monthly["date"], monthly["fatalities"], color="#C0392B", linewidth=1.5, zorder=3)

    for (start, end), color in zip(PERIOD_DATE_RANGES, PERIOD_COLORS):
        ax1.axvspan(start, end, alpha=0.18, color=color, zorder=0)

    ymax = ax1.get_ylim()[1]
    for (start, end), (_, _, label) in zip(PERIOD_DATE_RANGES, PERIODS):
        mid = start + (end - start) / 2
        ax1.text(
            mid, ymax * 0.97,
            label.split(":")[0],
            ha="center", va="top", fontsize=9, fontweight="bold", color="#444444",
        )

    for b in [pd.Timestamp("2016-10-01"), pd.Timestamp("2021-02-01"), pd.Timestamp("2023-11-01")]:
        ax1.axvline(b, color="#333333", linewidth=1.0, linestyle="--", alpha=0.55, zorder=4)

    ax1.set_ylabel("Fatal events per month", color="#5B9BD5", labelpad=8)
    ax2.set_ylabel("Fatalities per month", color="#C0392B", labelpad=8)
    ax1.set_title(
        "Myanmar: Monthly Fatal Conflict Events — Period Boundaries",
        fontsize=12, pad=10,
    )

    handles = [
        mpatches.Patch(fc="#5B9BD5", alpha=0.7, label="Fatal events/month"),
        Line2D([0], [0], color="#C0392B", linewidth=1.5, label="Fatalities/month"),
    ] + [
        mpatches.Patch(fc=c, alpha=0.4, label=lbl.split(":", 1)[1].strip())
        for c, (_, _, lbl) in zip(PERIOD_COLORS, PERIODS)
    ]
    ax1.legend(handles=handles, loc="upper left", fontsize=8, framealpha=0.85)

    ax1.xaxis.set_major_locator(mdates.YearLocator())
    ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha="right")

    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  ✓ {out_path.name}")


def plot_actor_timeline(
    df: pd.DataFrame, top10: pd.DataFrame, colors: dict, out_path: Path
) -> None:
    top10_actors = top10["actor1"].tolist()
    sub = df[df["actor1"].isin(top10_actors)].copy()
    sub["month"] = sub["event_date"].dt.to_period("M")

    monthly = (
        sub.groupby(["month", "actor1"])
        .agg(events=("event_id_cnty", "count"))
        .reset_index()
    )
    pivot = monthly.pivot(index="month", columns="actor1", values="events").fillna(0)
    pivot.index = pivot.index.to_timestamp()

    fig, ax = plt.subplots(figsize=(14, 6))
    for actor in top10_actors:
        if actor in pivot.columns:
            ax.plot(
                pivot.index, pivot[actor],
                label=actor, color=colors[actor], linewidth=1.4, alpha=0.85,
            )

    ax.set_title("Myanmar: Monthly Fatal Events by Top-10 Actor", fontsize=12, pad=10)
    ax.set_ylabel("Fatal events per month")
    ax.set_ylim(bottom=0)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")
    ax.legend(loc="upper left", fontsize=7.5, framealpha=0.85, ncol=2)

    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  ✓ {out_path.name}")

In [ ]:
def write_ranking_report(
    top10: pd.DataFrame,
    period_df: pd.DataFrame,
    n_events_total: int,
    out_path: Path,
) -> None:
    lines = [
        "# Myanmar Conflict Actor Ranking",
        "",
        f"**Generated:** {pd.Timestamp.today().date()}  ",
        f"**Filter:** country=Myanmar, fatalities>0, geo_precision≤{GEO_THRESH}  ",
        f"**In-scope events:** {n_events_total:,}  ",
        "**Merges:** Military Forces of Myanmar (4 date-suffix variants), "
        "People's Defense Force (all district fragments)  ",
        "**Exclusions:** Unidentified Armed Group, Unidentified Anti-Coup Armed Group, "
        "Protesters, Rioters, Police Forces  ",
        "**Kept separate:** Pyu Saw Htee, Zero Guerrilla Force - Myingyan  ",
        "",
        "---",
        "",
        "## Global Top 10 (all periods combined)",
        "",
        "Importance score = (rank by events + rank by fatalities) / 2. "
        "Tiebreak: fatalities descending.",
        "",
        "| Rank | Actor | Events | Fatalities | Score |",
        "|:---:|---|---:|---:|:---:|",
    ]
    for _, row in top10.iterrows():
        lines.append(
            f"| {int(row['global_rank'])} | {row['actor1']} "
            f"| {int(row['events']):,} | {int(row['fatalities']):,} "
            f"| {row['importance_score']:.1f} |"
        )

    lines += ["", "---", ""]

    for _, _, label in PERIODS:
        p_df = period_df[period_df["period"] == label]
        lines += [
            f"## {label}",
            "",
            "| Rank | Actor | Events | Fatalities |",
            "|:---:|---|---:|---:|",
        ]
        for _, row in p_df.iterrows():
            lines.append(
                f"| {int(row['period_rank'])} | {row['actor1']} "
                f"| {int(row['events']):,} | {int(row['fatalities']):,} |"
            )
        lines += [""]

    out_path.write_text("\n".join(lines), encoding="utf-8")
    print(f"  ✓ {out_path.name}")

In [ ]:
def main_03():
    print("Loading Myanmar fatal events...")
    df_raw = load_myanmar(DATA_IN)
    print(f"  {len(df_raw):,} events  (fatalities>0, geo≤{GEO_THRESH})")

    df = apply_merges(df_raw)
    n_pdf_merged = (df["actor1"] == PDF_LABEL).sum()
    n_mil_merged = (df["actor1"] == MILITARY_LABEL).sum()
    df = apply_exclusions(df)
    print(f"  Military merged: {n_mil_merged:,} events → {MILITARY_LABEL!r}")
    print(f"  PDF merged:      {n_pdf_merged:,} events → {PDF_LABEL!r}")
    print(f"  After exclusions: {len(df):,} events")

    print("\nComputing importance rankings...")
    all_actors, top10 = importance_ranking(df)

    top10_actors = top10["actor1"].tolist()
    colors = {actor: TAB10[i] for i, actor in enumerate(top10_actors)}
    color_path = OUT_DIR_03 / "actor_colors.json"
    json.dump({k: list(v) for k, v in colors.items()}, color_path.open("w"), indent=2)
    print(f"  ✓ actor_colors.json ({len(colors)} actors)")

    print("\nComputing per-period rankings...")
    period_df = period_ranking(df)

    print("\nGenerating figures...")
    plot_period_proposal(df_raw, OUT_DIR_03 / "01_period_proposal.png")
    plot_actor_timeline(df, top10, colors, OUT_DIR_03 / "01_actor_timeline.png")

    print("\nWriting report and CSVs...")
    write_ranking_report(top10, period_df, len(df), OUT_DIR_03 / "01_actor_ranking.md")

    top10_csv = top10[["global_rank", "actor1", "events", "fatalities", "importance_score"]].copy()
    top10_csv.to_csv(OUT_DIR_03 / "01_actors_top10_global.csv", index=False)
    print(f"  ✓ 01_actors_top10_global.csv  ({len(top10_csv)} rows)")

    period_df.to_csv(OUT_DIR_03 / "01_actors_per_period.csv", index=False)
    print(f"  ✓ 01_actors_per_period.csv  ({len(period_df)} rows)")

    print("\n── Global Top 10 ──────────────────────────────────────────────────────")
    for _, row in top10.iterrows():
        print(
            f"  {int(row['global_rank']):2d}. {row['actor1']:<48}"
            f"  events={int(row['events']):>6,}"
            f"  fatalities={int(row['fatalities']):>7,}"
            f"  score={row['importance_score']:.1f}"
        )

    print("\n── Per-period top 3 ────────────────────────────────────────────────────")
    for _, _, label in PERIODS:
        p_df = period_df[period_df["period"] == label]
        print(f"  {label}")
        for _, row in p_df.iterrows():
            print(
                f"    {int(row['period_rank'])}. {row['actor1']}"
                f"  ({int(row['events']):,} events, {int(row['fatalities']):,} fatalities)"
            )

    print("\nDone.")


main_03()

## Step 4 — Monthly Conflict Map GIF (3-Month Rolling Window)

Each GIF frame shows the 3-month window ending in the displayed month. For each top-10 actor, all event points in the window are:

1. Buffered by 5 km in UTM coordinates (EPSG:32647 — isotropic metric projection for Myanmar)
2. Merged with `unary_union` into a single territory silhouette per actor per frame

This **buffer-dissolve** approach converts scattered point data into readable territorial signatures, showing where each actor was operationally active rather than a cloud of overlapping dots.

**Caching:** Reprojecting ~50 k events from WGS-84 to UTM takes ~2 seconds. The result is cached to `data/processed/myanmar_events_projected.parquet` so subsequent runs are instant.

**Natural Earth boundaries** are downloaded once from the AWS S3 mirror (~10 MB) and cached to `data/raw/boundaries/myanmar/`. Steps 5 and 6 read from the same cache directory.

In [ ]:
COLORS_IN = ROOT / "output" / "myanmar" / "actor_colors.json"
PROJ_CACHE = ROOT / "data" / "processed" / "myanmar_events_projected.parquet"
OUT_DIR_04 = ROOT / "output" / "myanmar"
FRAMES_DIR = OUT_DIR_04 / "02_monthly_frames"
GIF_PATH = OUT_DIR_04 / "02_monthly_conflict.gif"
NOTES_PATH = OUT_DIR_04 / "02_animation_notes.md"
LEGEND_PATH = OUT_DIR_04 / "02_actor_color_legend.png"

WINDOW_MONTHS = 3

BUFFER_M_04 = 5_000
CRS = "EPSG:32647"
FRAME_MS_04 = 500
FIG_W, FIG_H = 10, 8
FIG_DPI_04 = 100
DOT_SIZE = 8
BUFFER_ALPHA = 0.22
DOT_ALPHA = 0.80
GIF_SIZE_LIMIT_MB = 30

NE_BASE = "https://naturalearth.s3.amazonaws.com/10m_cultural"
NE_URLS = {
    "ne_10m_admin_0_countries": f"{NE_BASE}/ne_10m_admin_0_countries.zip",
    "ne_10m_admin_1_states_provinces": f"{NE_BASE}/ne_10m_admin_1_states_provinces.zip",
}

In [ ]:
def _download_ne(name: str, url: str, dest: Path) -> Path:
    shp = dest / f"{name}.shp"
    if shp.exists():
        return shp
    print(f"  Downloading {name} from Natural Earth...")
    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
        z.extractall(dest)
    return shp


def load_boundaries_04(boundary_dir: Path, crs: str):
    """Download (first run) or load Myanmar boundaries from Natural Earth."""
    boundary_dir.mkdir(parents=True, exist_ok=True)
    a0_shp = _download_ne("ne_10m_admin_0_countries", NE_URLS["ne_10m_admin_0_countries"], boundary_dir)
    a1_shp = _download_ne("ne_10m_admin_1_states_provinces", NE_URLS["ne_10m_admin_1_states_provinces"], boundary_dir)

    a0 = gpd.read_file(a0_shp)
    a1 = gpd.read_file(a1_shp)

    mmr0 = a0[a0["NAME"].str.contains("Myanmar|Burma", na=False, case=False)].to_crs(crs)
    mmr1 = a1[a1["admin"].str.contains("Myanmar|Burma", na=False, case=False)].to_crs(crs)

    if mmr0.empty:
        mmr0 = a0[a0["SOVEREIGNT"].str.contains("Myanmar|Burma", na=False, case=False)].to_crs(crs)
    if mmr0.empty:
        raise ValueError("Myanmar admin0 not found. Available NAME values: " + str(a0["NAME"].unique()[:20]))

    print(f"  ✓ admin0: {len(mmr0)} feature(s)  |  admin1: {len(mmr1)} feature(s)")
    return mmr0, mmr1


def load_events_04(data_in: Path, cache: Path, actor_colors: dict, crs: str) -> gpd.GeoDataFrame:
    if cache.exists():
        print("  Loading projected events from cache...")
        return gpd.read_parquet(cache)

    print("  Projecting events (first run — will cache)...")
    df = pd.read_parquet(data_in)
    df = df[
        (df["country"] == COUNTRY)
        & (df["fatalities"] > 0)
        & (df["geo_precision"] <= GEO_THRESH)
    ].copy()
    df["event_date"] = pd.to_datetime(df["event_date"])

    df.loc[df["actor1"].isin(MILITARY_VARIANTS), "actor1"] = MILITARY_LABEL
    pdf_mask = (
        df["actor1"].str.contains("People's Defense Force", na=False)
        | df["actor1"].str.startswith("PDF:", na=False)
    )
    df.loc[pdf_mask, "actor1"] = PDF_LABEL
    df = df[~df["actor1"].isin(EXCLUDE)]
    df = df[df["actor1"].isin(actor_colors)]

    gdf = gpd.GeoDataFrame(
        df[["event_id_cnty", "event_date", "actor1", "fatalities"]],
        geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
        crs="EPSG:4326",
    ).to_crs(crs)

    gdf.to_parquet(cache)
    print(f"  ✓ Cached {len(gdf):,} projected events → {cache.name}")
    return gdf


def map_extent(mmr0: gpd.GeoDataFrame, pad: float = 0.04):
    minx, miny, maxx, maxy = mmr0.total_bounds
    dx, dy = maxx - minx, maxy - miny
    return minx - dx * pad, maxx + dx * pad, miny - dy * pad, maxy + dy * pad


def save_legend(actor_colors: dict, out_path: Path) -> None:
    n = len(actor_colors)
    fig, ax = plt.subplots(figsize=(5, n * 0.38 + 0.6))
    ax.set_axis_off()
    handles = [mpatches.Patch(color=c, label=a) for a, c in actor_colors.items()]
    ax.legend(handles=handles, loc="center", fontsize=8.5, framealpha=0.0)
    fig.tight_layout()
    fig.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close(fig)


def render_frame_04(
    month_str: str,
    window_gdf: gpd.GeoDataFrame,
    actor_colors: dict,
    mmr0: gpd.GeoDataFrame,
    mmr1: gpd.GeoDataFrame,
    extent: tuple,
    out_path: Path,
) -> None:
    fig, ax = plt.subplots(figsize=(FIG_W, FIG_H), dpi=FIG_DPI_04)
    ax.set_facecolor("#D6E8F7")

    mmr0.plot(ax=ax, facecolor="#F4F1EC", edgecolor="#333333", linewidth=1.0, zorder=1)
    mmr1.plot(ax=ax, facecolor="none", edgecolor="#AAAAAA", linewidth=0.35, zorder=2)

    for actor, color in actor_colors.items():
        sub = window_gdf[window_gdf["actor1"] == actor]
        if sub.empty:
            continue
        buf = gpd.GeoSeries(sub.geometry.buffer(BUFFER_M_04), crs=CRS).union_all()
        gpd.GeoSeries([buf], crs=CRS).plot(
            ax=ax, color=color, alpha=BUFFER_ALPHA, zorder=3
        )
        ax.scatter(
            sub.geometry.x, sub.geometry.y,
            color=color, s=DOT_SIZE, alpha=DOT_ALPHA, linewidths=0, zorder=4,
        )

    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    ax.set_axis_off()

    ax.text(
        0.02, 0.98, month_str,
        transform=ax.transAxes, fontsize=15, fontweight="bold",
        va="top", ha="left", color="#111111",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.82, edgecolor="none"),
        zorder=5,
    )
    ax.text(
        0.02, 0.91, f"n = {len(window_gdf):,} events",
        transform=ax.transAxes, fontsize=8.5,
        va="top", ha="left", color="#333333",
        bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.70, edgecolor="none"),
        zorder=5,
    )

    handles = [
        mpatches.Patch(color=c, alpha=0.8, label=a)
        for a, c in actor_colors.items()
    ]
    ax.legend(
        handles=handles, loc="lower right", fontsize=6,
        framealpha=0.85, title="Actor", title_fontsize=6.5,
    )

    fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
    fig.savefig(out_path, dpi=FIG_DPI_04, bbox_inches="tight")
    plt.close(fig)

In [ ]:
def main_04():
    OUT_DIR_04.mkdir(parents=True, exist_ok=True)
    FRAMES_DIR.mkdir(parents=True, exist_ok=True)

    print("Loading actor colors...")
    with COLORS_IN.open() as f:
        raw = json.load(f)
    actor_colors = {k: tuple(v) for k, v in raw.items()}
    print(f"  {len(actor_colors)} actors")

    print("\nLoading boundaries...")
    mmr0, mmr1 = load_boundaries_04(BOUNDARY_DIR, CRS)

    print("\nLoading events...")
    gdf = load_events_04(DATA_IN, PROJ_CACHE, actor_colors, CRS)
    print(f"  {len(gdf):,} top-10 actor events in scope")

    extent = map_extent(mmr0)

    months = pd.period_range(
        gdf["event_date"].dt.to_period("M").min(),
        gdf["event_date"].dt.to_period("M").max(),
        freq="M",
    )
    print(f"\nRendering {len(months)} frames ({months[0]} → {months[-1]})...")

    frame_paths = []
    skipped = []

    for i, month in enumerate(months):
        month_str = str(month)
        png_path = FRAMES_DIR / f"{month_str}.png"

        window_end = month.to_timestamp(how="end")
        window_start = (month - (WINDOW_MONTHS - 1)).to_timestamp()
        window_gdf = gdf[
            (gdf["event_date"] >= window_start) & (gdf["event_date"] <= window_end)
        ]

        if window_gdf.empty:
            skipped.append(month_str)
            continue

        render_frame_04(month_str, window_gdf, actor_colors, mmr0, mmr1, extent, png_path)
        frame_paths.append(png_path)

        if (i + 1) % 24 == 0 or i == len(months) - 1:
            pct = (i + 1) / len(months) * 100
            print(f"  [{pct:4.0f}%] frame {i+1}/{len(months)}  ({month_str})")

    print(f"\n  Rendered: {len(frame_paths)} | Skipped: {len(skipped)}")

    print("\nSaving legend...")
    save_legend(actor_colors, LEGEND_PATH)

    print("\nAssembling GIF...")
    frames_arr = [iio.imread(str(p)) for p in frame_paths]
    iio.mimwrite(str(GIF_PATH), frames_arr, duration=FRAME_MS_04 / 1000, loop=0)
    size_mb = GIF_PATH.stat().st_size / (1024 ** 2)
    print(f"  ✓ {GIF_PATH.name}  ({size_mb:.1f} MB)")
    if size_mb > GIF_SIZE_LIMIT_MB:
        print(f"  ⚠ Exceeds {GIF_SIZE_LIMIT_MB} MB — see animation notes for MP4 conversion")

    print("\nWriting animation notes...")
    lines = [
        "# Myanmar Monthly Conflict GIF — Animation Notes",
        "",
        f"**Generated:** {date.today()}  ",
        f"**Window:** {WINDOW_MONTHS}-month rolling (each frame covers [M-2 months, M])  ",
        f"**Actors:** top-10 by importance score (colors locked in actor_colors.json)  ",
        f"**Buffer:** {BUFFER_M_04 // 1000} km unary-union per actor per frame  ",
        f"**CRS:** {CRS}  ",
        f"**Frame duration:** {FRAME_MS_04} ms ({1000 / FRAME_MS_04:.1f} fps)  ",
        f"**Resolution:** {FIG_W * FIG_DPI_04} × {FIG_H * FIG_DPI_04} px  ",
        "",
        "## Frame statistics",
        "",
        f"- Month range: {months[0]} → {months[-1]} ({len(months)} months)",
        f"- Rendered frames: {len(frame_paths)}",
        f"- Skipped (empty window): {len(skipped)}",
        f"- GIF file size: {size_mb:.1f} MB",
    ]
    if size_mb > GIF_SIZE_LIMIT_MB:
        lines += [
            "",
            "## MP4 conversion (recommended — smaller file)",
            "",
            "```bash",
            "ffmpeg -i output/myanmar/02_monthly_conflict.gif \\",
            "  -vf 'fps=2,scale=trunc(iw/2)*2:trunc(ih/2)*2' \\",
            "  -movflags faststart output/myanmar/02_monthly_conflict.mp4",
            "```",
        ]
    if skipped:
        lines += ["", "## Skipped months", ""] + [f"- {m}" for m in skipped]
    NOTES_PATH.write_text("\n".join(lines), encoding="utf-8")
    print(f"  ✓ {NOTES_PATH.name}")

    print("\nDone.")


main_04()

## Kikuta (2022) OCSVM Module

[Kikuta (2022)](https://doi.org/10.1017/S0022381622000081) estimates conflict *zones* — contiguous geographic regions where armed groups were operationally active — using a **One-Class SVM (OCSVM)** with an RBF kernel. Unlike density clustering (DBSCAN, KMeans), OCSVM learns a decision boundary from positive examples only (conflict events), without needing labelled "non-conflict" areas.

**How it works:** Given a set of geolocated conflict events, the OCSVM finds the smallest hypersphere (in kernel feature space) that encloses most of the data. Grid cells projected inside the hypersphere boundary form the conflict zone.

**Key hyperparameters:**
- **gamma** (RBF kernel bandwidth): controls how tightly the zone wraps around clusters. Estimated via the *median heuristic*: `gamma = 1 / median(||xi - xj||²)` over a random subsample of standardized coordinates.
- **nu** (expected outlier fraction): estimated iteratively following Ghafoori et al. (2018) — start at 0.10, fit, measure actual outlier fraction, repeat until convergence (max 5 iterations, tolerance 0.01).

**Deviations from the paper** (SI 1 was not available in the published PDF):

| # | Paper | This replication | Reason |
|---|-------|-----------------|--------|
| 1 | Single OCSVM with date as 3rd feature | 12-month rolling window, no date feature | Avoids future-data contamination |
| 2 | gamma via SI 1 | Median heuristic | SI 1 unavailable |
| 3 | nu via SI 1 | Ghafoori iterative | Same approximation |
| 4 | Fatality weighting (method unstated) | Row replication, cap=99th percentile | sklearn OCSVM has no `sample_weight` |
| 5 | Min events: ≥4 (paper) | ≥10 | Avoids degenerate fits |
| 6 | EPSG:4326 degrees | EPSG:32647 km | Isotropic coordinates improve RBF distance |

The module is inlined directly (no `sys.path` manipulation needed).

In [ ]:
MIN_EVENTS = 10
_SEED = 20260428


def select_gamma(X_scaled: np.ndarray, n_subsample: int = 1000) -> float:
    """
    Median heuristic: gamma = 1 / median(||xi - xj||^2) over a random subsample.
    Approximates Ghafoori et al. (2018) initialiser (SI 1 unavailable).
    Input must already be StandardScaler-transformed.
    """
    n = len(X_scaled)
    if n <= 1:
        return 1.0
    rng = np.random.default_rng(_SEED)
    idx = rng.choice(n, size=min(n, n_subsample), replace=False)
    sub = X_scaled[idx]
    diff = sub[:, None, :] - sub[None, :, :]
    sq = (diff ** 2).sum(axis=-1)
    upper = sq[np.triu_indices(len(sub), k=1)]
    med = float(np.median(upper))
    return (1.0 / med) if med > 0 else (1.0 / max(X_scaled.shape[1], 1))


def fit_ocsvm_ghafoori(
    X_scaled: np.ndarray,
    gamma: float,
    nu0: float = 0.10,
    max_iter: int = 5,
    tol: float = 0.01,
) -> tuple:
    """
    Ghafoori et al. (2018) iterative nu estimation (approximated, max 5 iter).
    Adjusts nu until actual outlier fraction ≈ nu (self-consistent).
    Returns (fitted OneClassSVM, final_nu).
    """
    nu = float(np.clip(nu0, 0.01, 0.50))
    model = None
    for _ in range(max_iter):
        model = OneClassSVM(kernel="rbf", gamma=gamma, nu=nu)
        model.fit(X_scaled)
        outlier_frac = float(np.mean(model.predict(X_scaled) == -1))
        new_nu = float(np.clip(outlier_frac, 0.01, 0.50))
        if abs(new_nu - nu) < tol:
            nu = new_nu
            break
        nu = new_nu
    return model, nu


def _build_training_array(events: gpd.GeoDataFrame, fat_cap: int) -> np.ndarray:
    reps = np.clip(
        np.ceil(events["fatalities"].values).astype(int), 1, fat_cap
    )
    xy_km = np.column_stack([
        events.geometry.x.values / 1000.0,
        events.geometry.y.values / 1000.0,
    ])
    return np.repeat(xy_km, reps, axis=0)


def _mask_to_polygon(
    inside_mask: np.ndarray,
    xs_km: np.ndarray,
    ys_km: np.ndarray,
):
    dx = float(xs_km[1] - xs_km[0]) if len(xs_km) > 1 else 10.0
    dy = float(ys_km[1] - ys_km[0]) if len(ys_km) > 1 else 10.0
    rows, cols = np.where(inside_mask)
    if len(rows) == 0:
        return None
    boxes = [
        box(
            xs_km[c] - dx / 2, ys_km[r] - dy / 2,
            xs_km[c] + dx / 2, ys_km[r] + dy / 2,
        )
        for r, c in zip(rows, cols)
    ]
    merged = unary_union(boxes)
    return merged if not merged.is_empty else None


def compute_zone(
    events: gpd.GeoDataFrame,
    grid_xs_km: np.ndarray,
    grid_ys_km: np.ndarray,
    fat_cap: int,
    gamma_override: Optional[float] = None,
    nu_override: Optional[float] = None,
) -> tuple:
    """
    Fit OCSVM on events; evaluate on prediction grid; return zone polygon + metadata.

    Parameters
    ----------
    events        : GeoDataFrame with EPSG:32647 geometry and 'fatalities' column
    grid_xs_km    : 1D array, x-grid centres in km (west to east)
    grid_ys_km    : 1D array, y-grid centres in km (south to north)
    fat_cap       : maximum fatality replication count per event
    gamma_override, nu_override : bypass Ghafoori estimation if provided

    Returns
    -------
    zone_km : Shapely geometry in km (multiply x 1000 for metres) | None
    meta    : dict — gamma, nu, n_events, n_weighted, n_sv, outlier_frac, status
    """
    meta: dict = {
        "n_events": len(events),
        "n_weighted": 0,
        "gamma": None,
        "nu": None,
        "n_sv": None,
        "outlier_frac": None,
        "status": "ok",
    }

    if len(events) < MIN_EVENTS:
        meta["status"] = f"skip_n={len(events)}<{MIN_EVENTS}"
        return None, meta

    X_raw = _build_training_array(events, fat_cap)
    meta["n_weighted"] = int(len(X_raw))

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw)

    gamma = float(gamma_override) if gamma_override is not None else select_gamma(X_scaled)
    meta["gamma"] = gamma

    if nu_override is not None:
        nu = float(np.clip(nu_override, 0.01, 0.50))
        model = OneClassSVM(kernel="rbf", gamma=gamma, nu=nu)
        model.fit(X_scaled)
    else:
        model, nu = fit_ocsvm_ghafoori(X_scaled, gamma)
    meta["nu"] = float(nu)
    meta["n_sv"] = int(len(model.support_vectors_))
    meta["outlier_frac"] = float(np.mean(model.predict(X_scaled) == -1))

    XX, YY = np.meshgrid(grid_xs_km, grid_ys_km)
    grid_flat = np.column_stack([XX.ravel(), YY.ravel()])
    grid_scaled = scaler.transform(grid_flat)
    inside = (model.decision_function(grid_scaled) >= 0).reshape(XX.shape)

    zone_km = _mask_to_polygon(inside, grid_xs_km, grid_ys_km)
    if zone_km is None:
        meta["status"] = "empty_zone"

    return zone_km, meta


def _synthetic_events(cx_km, cy_km, n, spread_km, seed):
    rng = np.random.default_rng(seed)
    xy = rng.normal(loc=[cx_km, cy_km], scale=spread_km, size=(n, 2))
    pts = gpd.points_from_xy(xy[:, 0] * 1000, xy[:, 1] * 1000)
    return gpd.GeoDataFrame(
        {"fatalities": np.ones(n, dtype=int)},
        geometry=pts, crs="EPSG:32647",
    )


def sanity_cluster() -> None:
    """Zone must contain the centroid of a tight cluster of 80 events."""
    from shapely.geometry import Point
    cx, cy = 500.0, 2000.0
    events = _synthetic_events(cx, cy, n=80, spread_km=15, seed=1)
    xs = np.arange(200.0, 800.0, 10.0)
    ys = np.arange(1700.0, 2300.0, 10.0)
    zone_km, meta = compute_zone(events, xs, ys, fat_cap=10)
    assert zone_km is not None, f"sanity_cluster: zone is None — {meta['status']}"
    assert zone_km.contains(Point(cx, cy)), (
        f"sanity_cluster: zone does not contain cluster centroid — status={meta['status']}"
    )
    print(f"  sanity_cluster PASS  gamma={meta['gamma']:.4f}  nu={meta['nu']:.3f}  "
          f"n_sv={meta['n_sv']}  outlier_frac={meta['outlier_frac']:.3f}")


def sanity_outlier() -> None:
    """Zone must not extend 400 km into completely empty space."""
    from shapely.geometry import Point
    events = _synthetic_events(500.0, 2000.0, n=80, spread_km=10, seed=99)
    xs = np.arange(200.0, 1000.0, 10.0)
    ys = np.arange(1700.0, 2300.0, 10.0)
    zone_km, meta = compute_zone(events, xs, ys, fat_cap=10)
    assert zone_km is not None, f"sanity_outlier: zone is None — {meta['status']}"
    far_point = Point(900.0, 2000.0)
    assert not zone_km.contains(far_point), (
        f"sanity_outlier: zone extends 400 km into empty space — zone too large  "
        f"status={meta['status']}"
    )
    print(f"  sanity_outlier PASS  zone does not reach 400 km into empty space  nu={meta['nu']:.3f}")

## Step 5 — OCSVM Conflict-Zoning Pipeline

This step applies `compute_zone` to each **primary dyad** across every month in the dataset, using a 12-month rolling window.

**Why 12 months?** A longer window provides enough events per dyad for a stable OCSVM fit — many dyad-months have fewer than the `MIN_EVENTS = 10` threshold with a 3-month window, especially before 2021. Twelve months smooths seasonal variation while still capturing year-over-year shifts in territorial control.

**`WINDOW_MONTHS`** is reassigned here from 3 (Step 4's rolling GIF) to 12.

**Pipeline per dyad × month:**
1. Filter events to the dyad and 12-month window ending in that month
2. Replicate rows by fatality count (capped at 99th percentile) — heavier weighting for high-casualty events
3. StandardScale the (x km, y km) coordinates and run the OCSVM
4. Evaluate the decision function on a 10 km × 10 km prediction grid covering Myanmar
5. Merge inside-boundary grid cells into a WKT polygon stored in km coordinates

Results are saved per-dyad as parquet files in `data/processed/kikuta_zones/`.

**`STATIC_ONLY = False`** runs the full pipeline. Set to `True` to test a single dyad × month before committing to the full run (~15 min on a laptop).

### Primary Dyads and Prediction Grid

The four primary dyads all involve the Myanmar military as one party, reflecting the dominant conflict structure:

| Dyad key | Parties | Active since |
|----------|---------|--------------|
| `Military_vs_PDF` | Military vs People's Defense Force | 2021-02 (post-coup) |
| `Military_vs_KIO_KIA` | Military vs Kachin Independence Army | 2010 (ceasefire gaps) |
| `Military_vs_KNU_KNLA` | Military vs Karen National Liberation Army | 2010 (ongoing) |
| `Military_vs_ULA_AA` | Military vs Arakan Army | ~2019 (Rakhine offensive) |

The prediction grid covers Myanmar's bounding box at **10 km resolution** (EPSG:32647 km coordinates). Finer resolution increases computational cost quadratically; 10 km captures township-level territorial variation adequately for the OCSVM's spatial scale.

In [ ]:
STATIC_ONLY = False
STATIC_DYAD = "Military_vs_PDF"
STATIC_MONTH = "2022-07"

OUT_DIR_05 = ROOT / "output" / "myanmar" / "kikuta"

ATTACK_TYPES = frozenset({
    "Battles",
    "Explosions/Remote violence",
    "Violence against civilians",
})

PRIMARY_DYADS = [
    ("Military Forces of Myanmar (merged)", "People's Defense Force (merged)"),
    ("Military Forces of Myanmar (merged)", "KIO/KIA: Kachin Independence Organization/Kachin Independence Army"),
    ("Military Forces of Myanmar (merged)", "KNU/KNLA: Karen National Union/Karen National Liberation Army"),
    ("Military Forces of Myanmar (merged)", "ULA/AA: United League of Arakan/Arakan Army"),
]

GRID_RES_KM = 10.0
WINDOW_MONTHS = 12
FAT_CAP_QUANTILE = 0.99

In [ ]:
def load_and_prepare(data_in: Path) -> gpd.GeoDataFrame:
    """Load ACLED, apply Kikuta event filter + actor merges. Returns GeoDataFrame in EPSG:32647."""
    df = pd.read_parquet(data_in)
    df = df[
        (df["country"] == "Myanmar")
        & (df["fatalities"] > 0)
        & (df["geo_precision"] <= 2)
        & (df["event_type"].isin(ATTACK_TYPES))
    ].copy()
    df["event_date"] = pd.to_datetime(df["event_date"])

    df.loc[df["actor1"].isin(MILITARY_VARIANTS), "actor1"] = MILITARY_LABEL
    pdf1 = df["actor1"].str.contains("People's Defense Force", na=False) | \
           df["actor1"].str.startswith("PDF:", na=False)
    df.loc[pdf1, "actor1"] = PDF_LABEL

    df.loc[df["actor2"].isin(MILITARY_VARIANTS), "actor2"] = MILITARY_LABEL
    pdf2 = df["actor2"].str.contains("People's Defense Force", na=False) | \
           df["actor2"].str.startswith("PDF:", na=False)
    df.loc[pdf2, "actor2"] = PDF_LABEL

    df = df[df["actor2"].notna()]
    df = df[~df["actor1"].isin(EXCLUDE) & ~df["actor2"].isin(EXCLUDE)]

    def canonical_dyad(a1, a2):
        if a1 == MILITARY_LABEL:
            return a1, a2
        if a2 == MILITARY_LABEL:
            return a2, a1
        return (a1, a2) if a1 < a2 else (a2, a1)

    pairs = [canonical_dyad(r.actor1, r.actor2)
             for r in df[["actor1", "actor2"]].itertuples()]
    df["dyad_a1"] = [p[0] for p in pairs]
    df["dyad_a2"] = [p[1] for p in pairs]
    df["dyad"] = df["dyad_a1"] + " vs " + df["dyad_a2"]

    gdf = gpd.GeoDataFrame(
        df[["event_date", "actor1", "actor2", "dyad_a1", "dyad_a2", "dyad", "fatalities"]],
        geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
        crs="EPSG:4326",
    ).to_crs(CRS)

    return gdf


def build_myanmar_grid(gdf: gpd.GeoDataFrame):
    minx, miny, maxx, maxy = gdf.total_bounds
    pad = GRID_RES_KM * 1000 * 2
    xs_km = np.arange((minx - pad) / 1000, (maxx + pad) / 1000, GRID_RES_KM)
    ys_km = np.arange((miny - pad) / 1000, (maxy + pad) / 1000, GRID_RES_KM)
    return xs_km, ys_km


def dyad_key(a1: str, a2: str) -> str:
    short = {
        MILITARY_LABEL: "Military",
        PDF_LABEL: "PDF",
        "KIO/KIA: Kachin Independence Organization/Kachin Independence Army": "KIO_KIA",
        "KNU/KNLA: Karen National Union/Karen National Liberation Army": "KNU_KNLA",
        "ULA/AA: United League of Arakan/Arakan Army": "ULA_AA",
    }
    return f"{short.get(a1, a1[:20])}_vs_{short.get(a2, a2[:20])}"


def load_boundaries_disk(boundary_dir: Path, crs: str):
    """Load boundaries from disk (assumes Step 4 already downloaded them)."""
    a0_shp = boundary_dir / "ne_10m_admin_0_countries.shp"
    a1_shp = boundary_dir / "ne_10m_admin_1_states_provinces.shp"
    if not a0_shp.exists():
        raise FileNotFoundError(
            f"Boundary files not found in {boundary_dir}. Run Step 4 section first."
        )
    a0 = gpd.read_file(a0_shp)
    a1 = gpd.read_file(a1_shp)
    mmr0 = a0[a0["NAME"].str.contains("Myanmar|Burma", na=False, case=False)].to_crs(crs)
    mmr1 = a1[a1["admin"].str.contains("Myanmar|Burma", na=False, case=False)].to_crs(crs)
    if mmr0.empty:
        mmr0 = a0[a0["SOVEREIGNT"].str.contains("Myanmar|Burma", na=False, case=False)].to_crs(crs)
    return mmr0, mmr1


def plot_static(
    zone_km,
    events_window: gpd.GeoDataFrame,
    mmr0: gpd.GeoDataFrame,
    mmr1: gpd.GeoDataFrame,
    meta: dict,
    dyad_label: str,
    month_str: str,
    out_path: Path,
) -> None:
    fig, ax = plt.subplots(figsize=(9, 11), dpi=120)
    ax.set_facecolor("#D6E8F7")

    mmr0.plot(ax=ax, facecolor="#F4F1EC", edgecolor="#333333", linewidth=1.0, zorder=1)
    mmr1.plot(ax=ax, facecolor="none", edgecolor="#AAAAAA", linewidth=0.35, zorder=2)

    if zone_km is not None:
        zone_m_geom = sa.scale(zone_km, xfact=1000, yfact=1000, origin=(0, 0))
        gpd.GeoSeries([zone_m_geom], crs=CRS).plot(
            ax=ax, facecolor="#E87040", edgecolor="#AA3300",
            alpha=0.35, linewidth=0.8, zorder=3,
        )

    ax.scatter(
        events_window.geometry.x, events_window.geometry.y,
        color="#CC2200", s=12, alpha=0.65, linewidths=0, zorder=4,
    )

    minx, miny, maxx, maxy = mmr0.total_bounds
    dx, dy = maxx - minx, maxy - miny
    ax.set_xlim(minx - dx * 0.03, maxx + dx * 0.03)
    ax.set_ylim(miny - dy * 0.03, maxy + dy * 0.03)
    ax.set_axis_off()

    title = f"{dyad_label}\n12-month window ending {month_str}"
    ax.set_title(title, fontsize=11, fontweight="bold", pad=8)

    info = (
        f"n_events={meta['n_events']}  n_weighted={meta['n_weighted']}\n"
        f"gamma={meta['gamma']:.4f}  nu={meta['nu']:.3f}  "
        f"n_sv={meta['n_sv']}  outlier_frac={meta['outlier_frac']:.3f}\n"
        f"status={meta['status']}"
    )
    ax.text(
        0.02, 0.02, info,
        transform=ax.transAxes, fontsize=7.5, va="bottom", ha="left",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.80, edgecolor="none"),
        zorder=5,
    )

    patches = [
        mpatches.Patch(facecolor="#E87040", edgecolor="#AA3300", alpha=0.5, label="OCSVM zone"),
        mpatches.Patch(color="#CC2200", label=f"Events (n={meta['n_events']})"),
    ]
    ax.legend(handles=patches, loc="lower right", fontsize=8, framealpha=0.85)

    fig.tight_layout()
    fig.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    print(f"  ✓ {out_path.name}")

In [ ]:
def main_05():
    ZONES_DIR.mkdir(parents=True, exist_ok=True)
    OUT_DIR_05.mkdir(parents=True, exist_ok=True)

    print("Loading and preparing events...")
    gdf = load_and_prepare(DATA_IN)
    print(f"  {len(gdf):,} events after Kikuta filter (fatal, attack types, geo≤2)")

    fat_cap = int(np.ceil(np.quantile(gdf["fatalities"].values, FAT_CAP_QUANTILE)))
    print(f"  fat_cap = {fat_cap} (99th percentile of fatalities)")

    xs_km, ys_km = build_myanmar_grid(gdf)
    print(f"  Grid: {len(xs_km)} × {len(ys_km)} cells at {GRID_RES_KM} km")

    print("\nLoading boundaries...")
    mmr0, mmr1 = load_boundaries_disk(BOUNDARY_DIR, CRS)

    if STATIC_ONLY:
        print(f"\nSTATIC_ONLY mode: dyad={STATIC_DYAD}  month={STATIC_MONTH}")

        dyad_map = {dyad_key(a1, a2): (a1, a2) for a1, a2 in PRIMARY_DYADS}
        if STATIC_DYAD not in dyad_map:
            raise ValueError(f"Unknown STATIC_DYAD: {STATIC_DYAD}. Options: {list(dyad_map)}")
        a1, a2 = dyad_map[STATIC_DYAD]
        dyad_label = f"{a1.split('(')[0].strip()} vs {a2.split('(')[0].strip()}"

        month_end = pd.Period(STATIC_MONTH, freq="M").to_timestamp(how="end")
        month_start = (pd.Period(STATIC_MONTH, freq="M") - (WINDOW_MONTHS - 1)).to_timestamp()

        dyad_mask = (
            ((gdf["dyad_a1"] == a1) & (gdf["dyad_a2"] == a2)) |
            ((gdf["dyad_a1"] == a2) & (gdf["dyad_a2"] == a1))
        )
        events_window = gdf[
            dyad_mask
            & (gdf["event_date"] >= month_start)
            & (gdf["event_date"] <= month_end)
        ]

        print(f"  Window: {month_start.date()} → {month_end.date()}")
        print(f"  Events in window: {len(events_window)}")

        zone_km, meta = compute_zone(events_window, xs_km, ys_km, fat_cap)

        print(f"  gamma={meta['gamma']}  nu={meta['nu']}  n_sv={meta['n_sv']}  "
              f"outlier_frac={meta['outlier_frac']}  status={meta['status']}")

        out_path = OUT_DIR_05 / "03_zones_static.png"
        plot_static(zone_km, events_window, mmr0, mmr1, meta,
                    dyad_label, STATIC_MONTH, out_path)
        print("\nPause point 2: Review 03_zones_static.png before full run.")
        return

    print("\nFULL mode: computing all dyads × all months...")
    months = pd.period_range(
        gdf["event_date"].dt.to_period("M").min(),
        gdf["event_date"].dt.to_period("M").max(),
        freq="M",
    )
    print(f"  Month range: {months[0]} → {months[-1]} ({len(months)} months)")

    all_records = []
    skipped = []

    for a1, a2 in PRIMARY_DYADS:
        dk = dyad_key(a1, a2)
        dyad_label = f"{a1.split('(')[0].strip()} vs {a2.split('(')[0].strip()}"
        print(f"\n  Dyad: {dyad_label}")

        dyad_mask = (
            ((gdf["dyad_a1"] == a1) & (gdf["dyad_a2"] == a2)) |
            ((gdf["dyad_a1"] == a2) & (gdf["dyad_a2"] == a1))
        )
        dyad_gdf = gdf[dyad_mask]

        records = []
        for i, month in enumerate(months):
            month_end = month.to_timestamp(how="end")
            month_start = (month - (WINDOW_MONTHS - 1)).to_timestamp()
            events_window = dyad_gdf[
                (dyad_gdf["event_date"] >= month_start)
                & (dyad_gdf["event_date"] <= month_end)
            ]

            zone_km, meta = compute_zone(events_window, xs_km, ys_km, fat_cap)

            rec = {
                "dyad": dk,
                "dyad_a1": a1,
                "dyad_a2": a2,
                "month": str(month),
                "window_start": str(month_start.date()),
                "window_end": str(month_end.date()),
                **meta,
            }
            if zone_km is not None:
                rec["zone_wkt"] = zone_km.wkt
                rec["zone_area_km2"] = zone_km.area
            else:
                rec["zone_wkt"] = None
                rec["zone_area_km2"] = None
                skipped.append(f"{dk}/{month}")

            records.append(rec)

            if (i + 1) % 24 == 0 or i == len(months) - 1:
                pct = (i + 1) / len(months) * 100
                print(f"    [{pct:4.0f}%] {month}  status={meta['status']}")

        cache_path = ZONES_DIR / f"{dk}.parquet"
        pd.DataFrame(records).to_parquet(cache_path, index=False)
        print(f"    Saved → {cache_path.name}  ({len(records)} months, "
              f"{sum(1 for r in records if r['zone_wkt'])} zones)")
        all_records.extend(records)

    print(f"\n  Total zones computed: {sum(1 for r in all_records if r['zone_wkt'])}")
    print(f"  Skipped (< {MIN_EVENTS} events): {len(skipped)}")
    print("\nDone. Run Step 6 section to produce the animation.")


main_05()

## Step 6 — Kikuta Zone GIF Animation

Renders the zone polygons from Step 5 as animated GIFs:

- **4 individual GIFs** — one per dyad (`03_zones_Military_vs_PDF.gif`, etc.), zone shown in a distinct color
- **1 combined 4-panel GIF** (`03_zones_combined.gif`) — all four dyads side-by-side per frame, for direct comparison of territorial dynamics

**`FRAMES_DIR`** is reassigned to `03_frames` (distinct from Step 4's `02_monthly_frames`) so both animation pipelines coexist without overwriting each other.

**Zone coordinate system:** The OCSVM zones are stored in km coordinates in the parquet files. `zone_to_metres()` applies a 1000× scale transform before plotting in the EPSG:32647 (metre) CRS.

Each frame shows the zone for the 12-month window ending in the displayed month. Months where the dyad had fewer than `MIN_EVENTS = 10` events show no zone ("no zone" label).

In [ ]:
OUT_DIR_KIKUTA = ROOT / "output" / "myanmar" / "kikuta"
FRAMES_DIR = OUT_DIR_KIKUTA / "03_frames"

FRAME_MS = 400
FIG_DPI = 100
GIF_SIZE_LIMIT_MB_06 = 30

DYAD_CONFIG = [
    {
        "key": "Military_vs_PDF",
        "label": "Military vs PDF",
        "color": "#E05C30",
    },
    {
        "key": "Military_vs_KIO_KIA",
        "label": "Military vs KIO/KIA",
        "color": "#3A7DC9",
    },
    {
        "key": "Military_vs_KNU_KNLA",
        "label": "Military vs KNU/KNLA",
        "color": "#2CA05A",
    },
    {
        "key": "Military_vs_ULA_AA",
        "label": "Military vs ULA/AA",
        "color": "#9B59B6",
    },
]

In [ ]:
def load_boundaries_06(boundary_dir: Path, crs: str):
    a0_shp = boundary_dir / "ne_10m_admin_0_countries.shp"
    a1_shp = boundary_dir / "ne_10m_admin_1_states_provinces.shp"
    if not a0_shp.exists():
        raise FileNotFoundError(f"Boundaries missing — run Step 4 section first.")
    a0 = gpd.read_file(a0_shp)
    a1 = gpd.read_file(a1_shp)
    mmr0 = a0[a0["NAME"].str.contains("Myanmar|Burma", na=False, case=False)].to_crs(crs)
    mmr1 = a1[a1["admin"].str.contains("Myanmar|Burma", na=False, case=False)].to_crs(crs)
    if mmr0.empty:
        mmr0 = a0[a0["SOVEREIGNT"].str.contains("Myanmar|Burma", na=False, case=False)].to_crs(crs)
    return mmr0, mmr1


def map_extent_06(mmr0: gpd.GeoDataFrame, pad: float = 0.04):
    minx, miny, maxx, maxy = mmr0.total_bounds
    dx, dy = maxx - minx, maxy - miny
    return minx - dx * pad, maxx + dx * pad, miny - dy * pad, maxy + dy * pad


def zone_to_metres(wkt_str: str):
    geom_km = wkt.loads(wkt_str)
    return sa.scale(geom_km, xfact=1000, yfact=1000, origin=(0, 0))


def render_single_frame(
    month_str: str,
    zone_m,
    mmr0: gpd.GeoDataFrame,
    mmr1: gpd.GeoDataFrame,
    extent: tuple,
    color: str,
    label: str,
    meta: dict,
    out_path: Path,
) -> None:
    fig, ax = plt.subplots(figsize=(6, 9), dpi=FIG_DPI)
    ax.set_facecolor("#D6E8F7")
    mmr0.plot(ax=ax, facecolor="#F4F1EC", edgecolor="#333333", linewidth=0.8, zorder=1)
    mmr1.plot(ax=ax, facecolor="none", edgecolor="#AAAAAA", linewidth=0.3, zorder=2)

    if zone_m is not None:
        gpd.GeoSeries([zone_m], crs=CRS).plot(
            ax=ax, facecolor=color, edgecolor=color,
            alpha=0.40, linewidth=0.6, zorder=3,
        )

    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    ax.set_axis_off()

    ax.text(
        0.03, 0.98, month_str,
        transform=ax.transAxes, fontsize=13, fontweight="bold",
        va="top", ha="left", color="#111111",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.82, edgecolor="none"),
        zorder=5,
    )
    n_events = meta.get("n_events", 0)
    status = meta.get("status", "")
    info = f"n={n_events}" if status == "ok" else "no zone"
    ax.text(
        0.03, 0.91, info,
        transform=ax.transAxes, fontsize=8,
        va="top", ha="left", color="#444444",
        bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.70, edgecolor="none"),
        zorder=5,
    )
    ax.set_title(label, fontsize=9, fontweight="bold", pad=4)

    fig.subplots_adjust(left=0, right=1, bottom=0, top=0.96)
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)


def render_combined_frame(
    month_str: str,
    zones_m: list,
    metas: list,
    mmr0: gpd.GeoDataFrame,
    mmr1: gpd.GeoDataFrame,
    extent: tuple,
    out_path: Path,
) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(12, 16), dpi=FIG_DPI)
    axes = axes.flatten()

    for ax, cfg, zone_m, meta in zip(axes, DYAD_CONFIG, zones_m, metas):
        ax.set_facecolor("#D6E8F7")
        mmr0.plot(ax=ax, facecolor="#F4F1EC", edgecolor="#333333", linewidth=0.7, zorder=1)
        mmr1.plot(ax=ax, facecolor="none", edgecolor="#AAAAAA", linewidth=0.25, zorder=2)

        if zone_m is not None:
            gpd.GeoSeries([zone_m], crs=CRS).plot(
                ax=ax, facecolor=cfg["color"], edgecolor=cfg["color"],
                alpha=0.40, linewidth=0.5, zorder=3,
            )

        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])
        ax.set_axis_off()
        ax.set_title(cfg["label"], fontsize=9, fontweight="bold", pad=3)

        n_events = meta.get("n_events", 0)
        status = meta.get("status", "")
        info = f"n={n_events}" if status == "ok" else "no zone"
        ax.text(
            0.04, 0.04, info,
            transform=ax.transAxes, fontsize=7,
            va="bottom", ha="left", color="#555555",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.65, edgecolor="none"),
            zorder=5,
        )

    fig.suptitle(
        f"Myanmar Conflict Zones — {month_str}\n(12-month rolling window, OCSVM)",
        fontsize=12, fontweight="bold", y=0.995,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.985])
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)

In [ ]:
def main_06():
    OUT_DIR_KIKUTA.mkdir(parents=True, exist_ok=True)
    FRAMES_DIR.mkdir(parents=True, exist_ok=True)

    print("Loading boundaries...")
    mmr0, mmr1 = load_boundaries_06(BOUNDARY_DIR, CRS)
    extent = map_extent_06(mmr0)

    print("Loading zone data...")
    dyad_data = {}
    all_months = set()
    for cfg in DYAD_CONFIG:
        p = ZONES_DIR / f"{cfg['key']}.parquet"
        if not p.exists():
            raise FileNotFoundError(f"Missing {p} — run Step 5 section first.")
        df = pd.read_parquet(p)
        dyad_data[cfg["key"]] = df.set_index("month")
        all_months |= set(df["month"].tolist())
        n_zones = df["zone_wkt"].notna().sum()
        print(f"  {cfg['label']}: {n_zones} zones / {len(df)} months")

    months = sorted(all_months)
    print(f"\n  Total months: {len(months)}  ({months[0]} → {months[-1]})")

    gif_stats = {}
    for cfg in DYAD_CONFIG:
        key = cfg["key"]
        df_idx = dyad_data[key]
        label = cfg["label"]
        color = cfg["color"]
        dyad_frames_dir = FRAMES_DIR / key
        dyad_frames_dir.mkdir(exist_ok=True)

        print(f"\nRendering frames: {label}...")
        frame_paths = []
        for i, month_str in enumerate(months):
            out_path = dyad_frames_dir / f"{month_str}.png"
            if month_str in df_idx.index:
                row = df_idx.loc[month_str]
                meta = row.to_dict()
                zone_m = zone_to_metres(row["zone_wkt"]) if pd.notna(row.get("zone_wkt")) else None
            else:
                meta = {"n_events": 0, "status": "no_data"}
                zone_m = None

            render_single_frame(month_str, zone_m, mmr0, mmr1, extent,
                                 color, label, meta, out_path)
            frame_paths.append(out_path)

            if (i + 1) % 48 == 0 or i == len(months) - 1:
                print(f"  [{(i+1)/len(months)*100:4.0f}%] {month_str}")

        gif_path = OUT_DIR_KIKUTA / f"03_zones_{key}.gif"
        frames_arr = [iio.imread(str(p)) for p in frame_paths]
        iio.mimwrite(str(gif_path), frames_arr, duration=FRAME_MS / 1000, loop=0)
        size_mb = gif_path.stat().st_size / (1024 ** 2)
        gif_stats[key] = size_mb
        print(f"  ✓ {gif_path.name}  ({size_mb:.1f} MB)")

    print("\nRendering combined 4-panel frames...")
    combined_dir = FRAMES_DIR / "combined"
    combined_dir.mkdir(exist_ok=True)
    combined_paths = []

    for i, month_str in enumerate(months):
        out_path = combined_dir / f"{month_str}.png"
        zones_m = []
        metas = []
        for cfg in DYAD_CONFIG:
            df_idx = dyad_data[cfg["key"]]
            if month_str in df_idx.index:
                row = df_idx.loc[month_str]
                meta = row.to_dict()
                zone_m = zone_to_metres(row["zone_wkt"]) if pd.notna(row.get("zone_wkt")) else None
            else:
                meta = {"n_events": 0, "status": "no_data"}
                zone_m = None
            zones_m.append(zone_m)
            metas.append(meta)

        render_combined_frame(month_str, zones_m, metas, mmr0, mmr1, extent, out_path)
        combined_paths.append(out_path)

        if (i + 1) % 48 == 0 or i == len(months) - 1:
            print(f"  [{(i+1)/len(months)*100:4.0f}%] {month_str}")

    combined_gif = OUT_DIR_KIKUTA / "03_zones_combined.gif"
    frames_arr = [iio.imread(str(p)) for p in combined_paths]
    iio.mimwrite(str(combined_gif), frames_arr, duration=FRAME_MS / 1000, loop=0)
    combined_mb = combined_gif.stat().st_size / (1024 ** 2)
    print(f"  ✓ {combined_gif.name}  ({combined_mb:.1f} MB)")

    print("\nWriting replication notes...")
    zone_counts = {cfg["key"]: dyad_data[cfg["key"]]["zone_wkt"].notna().sum()
                   for cfg in DYAD_CONFIG}
    skip_counts = {cfg["key"]: (dyad_data[cfg["key"]]["zone_wkt"].isna()).sum()
                   for cfg in DYAD_CONFIG}

    lines = [
        "# Kikuta (2022) Replication — Notes",
        "",
        f"**Generated:** {date.today()}  ",
        "**Reference:** Kikuta (2022), OCSVM conflict-zoning method  ",
        "",
        "## Deviations from paper",
        "",
        "| # | Paper | This replication | Reason |",
        "|---|-------|-----------------|--------|",
        "| 1 | Single OCSVM with date as 3rd feature | 12-month rolling window, no date feature | Avoids future-data contamination |",
        "| 2 | gamma via SI 1 (unavailable) | Median heuristic: gamma = 1/median(||xi-xj||^2) | SI 1 not in PDF |",
        "| 3 | nu via SI 1 (unavailable) | Ghafoori iterative (max 5 iter, tol=0.01) | Same approximation |",
        "| 4 | Fatality weighting (method unstated) | Row replication, cap=99th pct | sklearn OneClassSVM has no sample_weight |",
        "| 5 | Minimum events: >=4 (paper) | >=10 | Stricter; avoids degenerate OCSVM fits |",
        "| 6 | EPSG:4326 degrees | EPSG:32647 km | Isotropic coordinates improve RBF kernel |",
        "",
        "## Zone counts per dyad",
        "",
        "| Dyad | Zones | Skipped | Notes |",
        "|------|-------|---------|-------|",
    ]
    for cfg in DYAD_CONFIG:
        k = cfg["key"]
        n_z = zone_counts[k]
        n_s = skip_counts[k]
        note = ""
        if k == "Military_vs_PDF":
            note = "Post-coup only; no events before 2021-02"
        elif k == "Military_vs_KIO_KIA":
            note = "Active 2011-2016, ceasefire gap, resumes 2021+"
        elif k == "Military_vs_KNU_KNLA":
            note = "Intensifies post-coup 2021+"
        elif k == "Military_vs_ULA_AA":
            note = "Intensifies from ~2019 (Rakhine offensive)"
        lines.append(f"| {cfg['label']} | {n_z} | {n_s} | {note} |")

    lines += [
        "",
        "## Parameters",
        "",
        f"- CRS: {CRS}  ",
        "- Grid: 10 km x 10 km  ",
        "- Window: 12-month rolling  ",
        "- gamma: median heuristic on standardized km coords  ",
        "- nu: Ghafoori iterative (nu0=0.10, max 5 iter, tol=0.01)  ",
        "- Fat cap: 99th percentile of fatalities in window  ",
        "- Min events per fit: 10  ",
        "- Event filter: fatal (fatalities>0), attack types (Battles, Explosions, VAC), geo_precision<=2  ",
    ]

    notes_path = OUT_DIR_KIKUTA / "03_replication_notes.md"
    notes_path.write_text("\n".join(lines), encoding="utf-8")
    print(f"  ✓ {notes_path.name}")

    print("\nDone.")


main_06()